In [1]:
import pandas as pd 
import numpy as np
df = pd.read_csv("my_data_q.csv")
df.head()

,quesNo,subjects,lokNo,member,ministry,type,date,sessionNo,supplementaryType,year
0,3008,"Toll Plazas in Dharmapuri, Tamil Nadu ...",18,Shri Mani A,ROAD TRANSPORT AND HIGHWAYS,UNSTARRED,2025-12-18,6,False,2025
1,2960,Programmes to improve Connectivity/Lo...,18,Shri Mani A,RAILWAYS,UNSTARRED,2025-12-17,6,False,2025
2,2570,Schemes to promote Skill Development among Far...,18,Shri Mani A,AGRICULTURE AND FARMERS WELFARE,UNSTARRED,2025-12-16,6,False,2025
3,2401,Contribution of States in Tourism Industry ...,18,Shri Mani A,TOURISM,UNSTARRED,2025-12-15,6,False,2025
4,2226,Pharmacovigilance System for AYUSH Drugs,18,"Shri E T Mohammed Basheer, Shri Mani A",AYUSH,UNSTARRED,2025-12-12,6,False,2025


In [3]:
texts = df["subjects"].fillna("").tolist()

In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
import faiss

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

embeddings = np.array(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

Batches:   0%|          | 0/1165 [00:00<?, ?it/s]

In [11]:
def recommend(question, k=5):

    vector = model.encode([question]).astype("float32")

    distances, indices = index.search(vector, k)

    return df.iloc[
        indices[0]
    ][[
        "subjects",
        "ministry",
        "member"
    ]]

In [12]:


recommend("solar energy")

,subjects,ministry,member
4180,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY,"Dr. Vinod Kumar Bind, Shri Pratap Chandra Sarangi"
27428,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY,"Dr. Vinod Kumar Bind, Shri Pratap Chandra Sarangi"
23327,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY,Shri Rajeev Rai
3741,Solar Energy Usage,NEW AND RENEWABLE ENERGY,"Shri Harish Chandra Meena, Shri Sukhdeo Bhagat"
17223,Solar Energy Usage,NEW AND RENEWABLE ENERGY,"Shri Harish Chandra Meena, Shri Sukhdeo Bhagat"


In [13]:
def recommend_by_id(row_number):

    query = df.iloc[row_number]["ministry"]

    vector = model.encode([query]).astype("float32")

    distances, indices = index.search(vector,6)

    return df.iloc[
        indices[0][1:]
    ]

In [14]:
recommend_by_id(100)

,quesNo,subjects,lokNo,member,ministry,type,date,sessionNo,supplementaryType,year
1784,2382,External Debt ...,18,Smt. Harsimrat Kaur Badal,FINANCE,UNSTARRED,2025-08-04,5,False,2025
22394,3140,Diplomatic Relations,18,Shri N K Premachandran,EXTERNAL AFFAIRS,UNSTARRED,2024-08-09,2,False,2024
1607,160,Country's External Debt,18,Shri Manickam Tagore B,FINANCE,UNSTARRED,2024-07-22,2,False,2024
23957,4771,Relations with Neighbouring Countries,18,Shri Amra Ram,EXTERNAL AFFAIRS,UNSTARRED,2025-03-28,4,False,2025
24607,4085,Corruption Cases,18,Shri Rajesh Ranjan,FINANCE,UNSTARRED,2025-08-18,5,False,2025


In [15]:
def search(query, k=5):
    query_vector = model.encode([query]).astype("float32")

    distances, indices = index.search(query_vector, k)

    results = df.iloc[indices[0]][
        ["subjects", "ministry", "member"]
    ].copy()

    results["distance"] = distances[0]

    return results

In [16]:
results = search("solar energy subsidy")

results

,subjects,ministry,member,distance
4180,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY,"Dr. Vinod Kumar Bind, Shri Pratap Chandra Sarangi",0.310962
23327,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY,Shri Rajeev Rai,0.310962
27428,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY,"Dr. Vinod Kumar Bind, Shri Pratap Chandra Sarangi",0.310962
1747,Solar Energy Policy,NEW AND RENEWABLE ENERGY,"Shri Patel Umeshbhai Babubhai, Shri Gajendra ...",0.328624
20748,Solar Energy Policy,NEW AND RENEWABLE ENERGY,"Shri Patel Umeshbhai Babubhai, Shri Gajendra ...",0.328624


In [18]:
df[df["ministry"].str.contains("HEALTH", case=False, na=False)].sample(10)


,quesNo,subjects,lokNo,member,ministry,type,date,sessionNo,supplementaryType,year
7334,2046,Ayushman Empaneled Hospitals in Andhra Pradesh,18,Shri Lavu Sri Krishna Devarayalu,HEALTH AND FAMILY WELFARE,UNSTARRED,2024-08-02,2,False,2024
5626,4318,Ambulances under the National Health Mission,18,Ms Iqra Choudhary,HEALTH AND FAMILY WELFARE,UNSTARRED,2024-12-20,3,False,2024
33041,3241,National Polio Surveillance Programme,18,"Smt. Kamaljeet Sehrawat, Shri Kanwar Singh Tan...",HEALTH AND FAMILY WELFARE,UNSTARRED,2025-08-08,5,False,2025
65,3055,National Urban Health Mission ...,18,Shri Mani A,HEALTH AND FAMILY WELFARE,UNSTARRED,2024-12-13,3,False,2024
8565,2015,Cases of Antimicrobial Resistance,18,Shri Vamsi Krishna Gaddam,HEALTH AND FAMILY WELFARE,UNSTARRED,2024-12-06,3,False,2024
3104,173,Coverage of Transgender Beneficiaries under A...,18,"Shri Benny Behanan, Shri K Sudhakaran",HEALTH AND FAMILY WELFARE,STARRED,2024-08-02,2,False,2024
7887,697,Increase of MBBS Seats in Medical College ...,18,"Dr. Nishikant Dubey, Shri Chavda Vinod Lakhams...",HEALTH AND FAMILY WELFARE,UNSTARRED,2024-11-29,3,False,2024
25886,1897,Vaccines to Children,18,Shri Magunta Sreenivasulu Reddy,HEALTH AND FAMILY WELFARE,UNSTARRED,2024-08-02,2,False,2024
5749,2083,National Midwifery Programme,18,Shri Darshan Singh Choudhary,HEALTH AND FAMILY WELFARE,UNSTARRED,2025-12-12,6,False,2025
868,2034,Eradication of Chronic Endemic Diseases,18,"Shri Selvam G, Shri C N Annadurai",HEALTH AND FAMILY WELFARE,UNSTARRED,2024-12-06,3,False,2024
